# ComfyUI for Paperspace (A6000)

This notebook is optimized for Paperspace Gradient notebooks with NVIDIA A6000 GPU.

**Session options:**
* Machine: A6000 (or similar NVIDIA GPU)
* Runtime: 6+ hours recommended
* Storage: Persistent storage enabled for models

# Installation

In [ ]:
%%time
update = False

import os
import stat

# Paperspace paths
home_dir = '/notebooks'
python = '/notebooks/venv/bin/python'
pip = '/notebooks/venv/bin/pip'

def find_bin_folders(folder_path):
    bin_folders = []
    for root, dirs, files in os.walk(folder_path):
        for dir_name in dirs:
            if dir_name == 'bin':
                bin_folders.append(os.path.join(root, dir_name)) 
    return bin_folders

def installLibraries(home_dir, python, pip):
    %cd {home_dir}
    # CUDA 12.1 for A6000 GPU
    !{pip} install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
    !{pip} install tensorflow[and-cuda]
    # Download requirements
    !wget -q https://q4j3.c11.e2-5.dev/downloads/req.txt -O /tmp/req.txt
    !{pip} install -r /tmp/req.txt

!pip install virtualenv

if not os.path.exists(f'{home_dir}/venv'):
    print('Installing virtual environment...')
    os.chdir(home_dir)
    get_ipython().system(f'cd {home_dir}')
    
    get_ipython().system('virtualenv venv -p $(which python3.10 || which python3)')
    installLibraries(home_dir, python, pip)
else:
    bin_folders = find_bin_folders('/notebooks/venv')
    if bin_folders:
        print("Found 'bin' folders:")
        for bin_folder in bin_folders:
            print(bin_folder)
            for filename in os.listdir(bin_folder):
                file_path = os.path.join(bin_folder, filename)
                if os.path.isfile(file_path):
                    current_permissions = os.stat(file_path).st_mode
                    # Add execute permissions for the user, group, and others
                    os.chmod(file_path, current_permissions | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

# Set up python symlinks
python_bin = '/notebooks/venv/bin'
if not os.path.exists(f'{python_bin}/python'):
    if os.path.exists(f'{python_bin}/python3.10'):
        !ln -sf {python_bin}/python3.10 {python_bin}/python
        !ln -sf {python_bin}/python3.10 {python_bin}/python3
    elif os.path.exists(f'{python_bin}/python3'):
        !ln -sf {python_bin}/python3 {python_bin}/python

# Clone ComfyUI
%cd /notebooks
if not os.path.exists('/notebooks/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
else:
    # Update existing installation to latest version
    %cd /notebooks/ComfyUI
    !git pull origin master
%cd /notebooks/ComfyUI

!{pip} install -r requirements.txt

# Set up temporary model storage
!mkdir -p /tmp/models/checkpoints
!mkdir -p /tmp/models/clip
!mkdir -p /tmp/models/vae
!mkdir -p /tmp/models/unet

# Remove default model folders and link to temp storage
# Comment out these lines if you want to keep models in persistent storage
!rm -rf /notebooks/ComfyUI/models/checkpoints
!rm -rf /notebooks/ComfyUI/models/clip
!rm -rf /notebooks/ComfyUI/models/vae
!rm -rf /notebooks/ComfyUI/models/unet

!ln -sf /tmp/models/checkpoints /notebooks/ComfyUI/models/checkpoints
!ln -sf /tmp/models/clip /notebooks/ComfyUI/models/clip
!ln -sf /tmp/models/vae /notebooks/ComfyUI/models/vae
!ln -sf /tmp/models/unet /notebooks/ComfyUI/models/unet

checkpoints = '/notebooks/ComfyUI/models/checkpoints'
loras = '/notebooks/ComfyUI/models/loras'
link_path = checkpoints + '/temp-models'
temp_models = '/tmp/temp-models'

!mkdir -p /tmp/temp-models

if not os.path.exists(link_path):
    get_ipython().system(f'ln -sf {temp_models} {checkpoints}')

# Install the node manager
update_manager = True
%cd /notebooks/ComfyUI/custom_nodes
if not os.path.exists('/notebooks/ComfyUI/custom_nodes/ComfyUI-Manager'):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git
%cd ComfyUI-Manager
if update_manager:
    get_ipython().system('git pull')

# Pinggy script for tunneling
!wget -q https://raw.githubusercontent.com/wandaweb/jupyter-webui-tunneling/main/pinggy.py -O /notebooks/pinggy.py

# Second GPU offload (optional, useful for dual-GPU setups)
%cd /notebooks/ComfyUI/custom_nodes
!wget -q https://gist.githubusercontent.com/city96/30743dfdfe129b331b5676a79c3a8a39/raw/ecb4f6f5202c20ea723186c93da308212ba04cfb/ComfyBootlegOffload.py

print('\n✅ Installation complete!')

---
# WebUI

## Start the WebUI with Pinggy
* Wait for the GUI to start.
* Click the link that ends with .pinggy.link 😁
* If generation is still running after the link expires in an hour, wait for the generation to complete and restart the WebUI code block to get a new link

In [ ]:
# Starting the Web UI with pinggy

%cd /notebooks/ComfyUI
!python /notebooks/pinggy.py --command='/notebooks/venv/bin/python /notebooks/ComfyUI/main.py' --port=8188

## Start the WebUI with Zrok

### Install Zrok

In [ ]:
# Install Zrok (only needs to run once)

!mkdir -p /notebooks/zrok
%cd /notebooks/zrok
!rm -f zrok*.gz
!wget -q https://github.com/openziti/zrok/releases/download/v1.1.10/zrok_1.1.10_linux_amd64.tar.gz
!tar -xvf ./zrok*.gz 
!chmod a+x /notebooks/zrok/zrok 

### Create a Zrok account
Enter your email address in the email variable

In [ ]:
email = '####@gmail.com'  # replace with your email

# --------------

cmd = '/notebooks/zrok/zrok invite'
log = '/notebooks/zrok/log.txt'

!pip install pexpect
!touch $log

import pexpect
import time
child = pexpect.spawn('bash')
child.sendline(f'{cmd} | tee {log}')
child.expect('enter and confirm your email address...')
time.sleep(1); child.sendline(email); time.sleep(1); child.send(chr(9)); time.sleep(1)
child.sendline(email); time.sleep(1); child.send('\n'); time.sleep(1); child.send(chr(9))
time.sleep(1); child.send('\r\n'); time.sleep(2); child.close()
!cat $log
!rm $log

### Enable Zrok
Paste your Zrok token in the token variable

In [ ]:
# Enable Zrok (needs to run once per instance)
# Paste your Zrok token in the token variable

token = ""
!chmod a+x /notebooks/zrok/zrok 
!/notebooks/zrok/zrok enable $token

### Start the WebUI with Zrok

In [ ]:
# Start the WebUI with Zrok
%cd /notebooks/ComfyUI
command = '/notebooks/venv/bin/python /notebooks/ComfyUI/main.py'
port = '8188'
# ------------------------

!chmod a+x /notebooks/zrok/zrok 
cmd = f'{command} & /notebooks/zrok/zrok share public http://localhost:{port} --headless'
get_ipython().system(cmd)

---
# Model Management

## Install a model

Copy the model URL to the model_url field. Make sure the model can be accessed publicly, without being signed into a website.

In [ ]:
# Install a model in permanent storage
model_url = 'https://civitai.com/api/download/models/782002'
model_name = 'JuggernautXL.safetensors'

%cd $checkpoints
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

In [ ]:
# Install a LoRA in permanent storage
model_url = 'https://civitai.com/api/download/models/137124?type=Model&format=SafeTensor'
model_name = 'DreamArt.safetensors'

%cd /notebooks/ComfyUI/models/loras
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

In [ ]:
# Install a model in temporary storage
model_url = 'https://civitai.com/api/download/models/456751'
model_name = 'HelloWorld-XL.safetensors' 

%cd $temp_models
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

## Download a model for a custom node

In [ ]:
model_folder = '/notebooks/ComfyUI/custom_nodes/my_node/models'
model_url = ''
model_name = 'model.safetensors'

%cd $model_folder
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

---
# File Browser

## Install FileBrowser

In [ ]:
%cd /notebooks
!wget -q https://github.com/filebrowser/filebrowser/releases/download/v2.57.0/linux-amd64-filebrowser.tar.gz
!tar xvfz linux-amd64-filebrowser.tar.gz
!chmod a+x /notebooks/filebrowser
!/notebooks/filebrowser config init 
!/notebooks/filebrowser config set --auth.method=noauth > /dev/null
!/notebooks/filebrowser config set --branding.theme=dark > /dev/null
!/notebooks/filebrowser users add admin admin 
!/notebooks/filebrowser config export "/notebooks/config.json"

## Run FileBrowser

In [ ]:
%cd /notebooks
!chmod a+x /notebooks/filebrowser

!python /notebooks/pinggy.py --command='/notebooks/filebrowser -c "/notebooks/config.json"' --port=8080

---
# Delete a model

In [ ]:
# List permanent models
!ls -la $checkpoints

# Delete a model
model_to_delete = '/notebooks/ComfyUI/models/checkpoints/model.safetensors'
!rm $model_to_delete

In [ ]:
# Check the size of a model (update the path to your model)
model_to_check = '/notebooks/ComfyUI/models/loras/DreamArt.safetensors'  # example
!du -sh $model_to_check

---
# Delete everything in the working folder

In [ ]:
# Delete the working folder
!rm -rf /notebooks/*